In [1]:
import os
os.chdir("..")

In [2]:
from src.services.arxiv.client import ArxivClient
from src.config import ArxivSettings
import json

arxiv_settings = ArxivSettings()

In [3]:
print(json.dumps(arxiv_settings.model_dump(), indent=4))

{
    "base_url": "https://export.arxiv.org/api/query",
    "pdf_cache_dir": "./data/arxiv_pdfs",
    "rate_limit_delay": 3.0,
    "timeout_seconds": 30,
    "max_results": 15,
    "search_category": "cs.AI",
    "download_max_retries": 3,
    "download_retry_delay_base": 5.0,
    "max_concurrent_downloads": 5,
    "max_concurrent_parsing": 1,
    "namespaces": {
        "atom": "http://www.w3.org/2005/Atom",
        "opensearch": "http://a9.com/-/spec/opensearch/1.1/",
        "arxiv": "http://arxiv.org/schemas/atom"
    }
}


In [4]:
arxiv_client = ArxivClient(settings=arxiv_settings)

In [5]:
arxiv_papers = await arxiv_client.fetch_papers()

In [6]:
arxiv_papers[0].model_dump()

{'arxiv_id': '2607.26057v1',
 'title': 'Pass the Baton: Trajectory-Relayed On-Policy Distillation',
 'authors': ['Haolei Xu',
  'Xiaowen Xu',
  'Haiwen Hong',
  'Zixuan Ni',
  'Hongxing Li',
  'Yiwen Qiu',
  'Weiming Lu',
  'Yongliang Shen'],
 'abstract': "On-policy distillation (OPD) grounds token-level supervision in the student's own trajectory, yet suffers from prefix failure: once the student commits to a wrong reasoning direction, all subsequent generation builds on this deviation, producing misdirected continuations that elicit unreliable supervision and waste compute. We identify a teacher-student continuation asymmetry on failed prefixes, where the teacher tends to redirect while the student continues along the original direction, and convert it into a label-free handoff trigger in Relay On-Policy Distillation (Relay-OPD). During training, Relay-OPD constructs relay trajectories by letting the teacher briefly take over at detected trigger points to produce a teacher leg, after

In [7]:
await arxiv_client.download_pdf(arxiv_papers[0])

PosixPath('data/arxiv_pdfs/2607.26057v1.pdf')

In [8]:
arxiv_papers[0].model_dump().get("abstract")

"On-policy distillation (OPD) grounds token-level supervision in the student's own trajectory, yet suffers from prefix failure: once the student commits to a wrong reasoning direction, all subsequent generation builds on this deviation, producing misdirected continuations that elicit unreliable supervision and waste compute. We identify a teacher-student continuation asymmetry on failed prefixes, where the teacher tends to redirect while the student continues along the original direction, and convert it into a label-free handoff trigger in Relay On-Policy Distillation (Relay-OPD). During training, Relay-OPD constructs relay trajectories by letting the teacher briefly take over at detected trigger points to produce a teacher leg, after which the student resumes and is optimized on the resulting trajectory. A limited relay budget concentrates intervention on critical early positions while limiting departure from the student policy. With a Qwen3-4B-Instruct-2507 teacher and Qwen3-0.6B/1.7

In [9]:
os.getcwd()

'/home/thienhb/Workspace/arxiv-paper-rag'

In [10]:
# Test arXiv API Client
import asyncio
from datetime import datetime, timedelta

# Import our arXiv client
from src.services.arxiv.factory import make_arxiv_client

print("TESTING ARXIV API CLIENT")
print("=" * 40)

# Create client
arxiv_client = make_arxiv_client()
print(f"✓ Client created: {arxiv_client.base_url}")
print(f"   Rate limit: {arxiv_client.rate_limit_delay}s")
print(f"   Max results: {arxiv_client.max_results}")
print(f"   Category: {arxiv_client.search_category}")
print()

TESTING ARXIV API CLIENT
✓ Client created: https://export.arxiv.org/api/query
   Rate limit: 3.0s
   Max results: 15
   Category: cs.AI



In [11]:
# Test Paper Fetching
async def test_paper_fetching():
    """Test fetching papers from arXiv with rate limiting."""
    
    print("Test 1: Fetch Recent CS.AI Papers")
    try:
        papers = await arxiv_client.fetch_papers(
            max_results=2, 
            sort_by="submittedDate",
            sort_order="descending"
        )
        
        print(f"✓ Fetched {len(papers)} papers")
        
        if papers:
            for i, paper in enumerate(papers[:2], 1):
                print(f"   {i}. [{paper.arxiv_id}] {paper.title[:60]}...")
                print(f"      Authors: {', '.join(paper.authors[:2])}{'...' if len(paper.authors) > 2 else ''}")
                print(f"      Categories: {', '.join(paper.categories)}")
                print(f"      Published: {paper.published_date}")
                print()
        
        return papers
        
    except Exception as e:
        print(f"✗ Error fetching papers: {e}")
        if "503" in str(e):
            print("   arXiv API temporarily unavailable (normal)")
            print("   Rate limiting and error handling working correctly")
        return []

# Run the test
papers = await test_paper_fetching()


Test 1: Fetch Recent CS.AI Papers
✓ Fetched 2 papers
   1. [2607.26057v1] Pass the Baton: Trajectory-Relayed On-Policy Distillation...
      Authors: Haolei Xu, Xiaowen Xu...
      Categories: cs.CL, cs.AI
      Published: 2026-07-28T17:59:46Z

   2. [2607.26055v1] $π\mathbf{R}^2$: Reactive Real-time Flow Policies...
      Authors: Sungjae Park, Shubham Tulsiani
      Categories: cs.RO, cs.AI, cs.LG
      Published: 2026-07-28T17:59:31Z



In [12]:
# Test Date Filtering
async def test_date_filtering():
    """Test date range filtering functionality."""
    
    print("Test 2: Date Range Filtering")
    
    # Use specific dates: 
    from_date = "20250808"  
    to_date = "20250809"    
    try:
        date_papers = await arxiv_client.fetch_papers(
            max_results=5,
            from_date=from_date,
            to_date=to_date
        )
        
        print(f"✓ Date filtering test: {len(date_papers)} papers from {from_date}-{to_date}")
        
        if date_papers:
            for i, paper in enumerate(date_papers, 1):
                print(f"   {i}. [{paper.arxiv_id}] {paper.title[:60]}...")
                print(f"      Authors: {', '.join(paper.authors[:2])}{'...' if len(paper.authors) > 2 else ''}")
                print(f"      Categories: {', '.join(paper.categories)}")
                print(f"      Published: {paper.published_date}")
                print()
        
        return date_papers
        
    except Exception as e:
        print(f"✗ Date filtering error: {e}")
        return []

# Run date filtering test
date_papers = await test_date_filtering()


Test 2: Date Range Filtering
✓ Date filtering test: 5 papers from 20250808-20250809
   1. [2508.07111v1] Investigating Intersectional Bias in Large Language Models u...
      Authors: Falaah Arif Khan, Nivedha Sivakumar...
      Categories: cs.CL, cs.AI
      Published: 2025-08-09T22:24:40Z

   2. [2508.07107v2] Designing a Feedback-Driven Decision Support System for Dyna...
      Authors: Timothy Oluwapelumi Adeyemi, Nadiah Fahad AlOtaibi
      Categories: cs.AI, cs.CY
      Published: 2025-08-09T21:24:54Z

   3. [2508.07102v1] Towards High-Order Mean Flow Generative Models: Feasibility,...
      Authors: Yang Cao, Yubin Chen...
      Categories: cs.LG, cs.AI, cs.CV
      Published: 2025-08-09T21:10:58Z

   4. [2508.07101v2] Less Is More: Fast and Accurate Reasoning with Cross-Head Un...
      Authors: Lijie Yang, Zhihao Zhang...
      Categories: cs.CL, cs.AI
      Published: 2025-08-09T21:10:33Z

   5. [2508.07095v1] Hide or Highlight: Understanding the Impact of Factuality Ex...
  

In [13]:
# Test PDF Download
async def test_pdf_download(test_papers):
    """Test PDF downloading with caching."""

    print("Test 3: PDF Download & Caching")
    
    if not test_papers:
        print("No papers available for PDF download test")
        return None
    
    # Test with first paper
    test_paper = test_papers[0]
    print(f"Testing PDF download for: {test_paper.arxiv_id}")
    print(f"Title: {test_paper.title[:60]}...")
    
    try:
        # Download PDF 
        pdf_path = await arxiv_client.download_pdf(test_paper)
        
        if pdf_path and pdf_path.exists():
            size_mb = pdf_path.stat().st_size / (1024 * 1024)
            print(f"✓ PDF downloaded: {pdf_path.name} ({size_mb:.2f} MB)")
            
            return pdf_path
        else:
            print("✗ PDF download failed")
            return None
            
    except Exception as e:
        print(f"✗ PDF download error: {e}")
        return None

# Run PDF download test 
pdf_path = await test_pdf_download(date_papers[:1])

Test 3: PDF Download & Caching
Testing PDF download for: 2508.07111v1
Title: Investigating Intersectional Bias in Large Language Models u...
✓ PDF downloaded: 2508.07111v1.pdf (6.81 MB)
